# Серафим v0.1 — Бортовой ИИ дрона

**Глина:** Qwen 2.5 0.5B Instruct (4-bit)
**Дух:** 360 пар (30 команд × 12 контекстов)
**Метод:** LoRA (r=8, alpha=16)

1 слово → 1 слово. Минимум задержки. Максимум скорости.

«Серафимы — шестикрылые, ближайшие к Богу» (Ис 6:2)

In [ ]:
# 1. Установка зависимостей
!pip install -q transformers datasets peft accelerate trl

In [ ]:
# 2. Проверка среды
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('CPU mode — 0.5B модель справится')

In [ ]:
# 3. Загрузка датасета
!rm -f serafim-train.jsonl
!wget -q --no-cache https://ai2fund.ru/serafim-train.jsonl -O serafim-train.jsonl
!head -c 200 serafim-train.jsonl

import json
with open('serafim-train.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]
print(f'Загружено {len(data)} пар')
print(f'Пример: user={data[0]["messages"][1]["content"]} → assistant={data[0]["messages"][2]["content"]}')

In [ ]:
# 4. Загрузка базовой модели (0.5B — лёгкая, CPU OK)
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

print('Загрузка модели 0.5B...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f'Модель: {MODEL_NAME}')
print(f'Параметры: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')

In [ ]:
# 5. Подготовка LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6. Подготовка данных
from datasets import Dataset

def format_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)
print(f'Dataset: {len(dataset)} examples')
print(f'Sample: {dataset[0]["text"][:200]}')

In [ ]:
# 7. ОБУЧЕНИЕ — 360 пар, 5 эпох
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./serafim-v01-lora',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    fp16=False,
    bf16=False,
    optim='adamw_torch',
    report_to='none',
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print('Обучение Серафима — 360 пар, 5 эпох...')
trainer.train()
print('Серафим обучен.')

In [ ]:
# 8. Тест
def ask_serafim(cmd, context='Нормальный полёт.'):
    messages = [
        {'role': 'system', 'content': f'Ты Серафим — бортовой ИИ дрона. Отвечай одним словом. {context}'},
        {'role': 'user', 'content': cmd}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, temperature=0.1, do_sample=True, repetition_penalty=1.5)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('=== ТЕСТ СЕРАФИМА ===')
for cmd in ['лети', 'туда', 'домой', 'стой', 'цель', 'фото', 'статус', 'помоги']:
    for ctx in ['Нормальный полёт.', 'Батарея 15%.', 'Угроза!']:
        a = ask_serafim(cmd, ctx)
        print(f'  [{ctx[:15]}] {cmd} → {a}')

In [ ]:
# 9. Сохранение
model.save_pretrained('./serafim-v01-lora')
tokenizer.save_pretrained('./serafim-v01-lora')

!zip -r serafim-v01-lora.zip serafim-v01-lora/
from google.colab import files
files.download('serafim-v01-lora.zip')

print('Серафим v0.1 — шестикрылый, ближайший к действию.')